In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score

import plotly.graph_objects as go

import fastplotlib as fpl

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


In [2]:
np.random.seed(42)

# Spring Chain

## Sim

In [3]:
time = 1000  # Total seconds
dt = 0.005
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

N = 10

m_diag = np.random.uniform(0.1, .5, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, .3, size=N))

k_vals = np.random.uniform(0.5, 8, N - 1)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals
K[idx + 1, idx] = -k_vals
k_padded = np.pad(k_vals, (1, 1), mode="constant")
K[np.arange(N), np.arange(N)] = k_padded[:-1] + k_padded[1:]


t = np.linspace(0, time, steps)

u_val = 1
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [4]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        # We apply the force of u to the first spring
        u_force = np.zeros(N)
        u_force[0] = u[i - 1]

        # Velocity Verlet
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * .5 * dt**2

        # Now using what values we just found we calc v
        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        # Do v[i - 1] + acc * dt to get v[i]
        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [6]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [ ]:
x_init = np.arange(N).astype(np.float32)  # Gonna have every spring be at 1, 2, 3
y_init = np.zeros(N).astype(np.float32)  # Start all dots at y=0

dots_data = np.column_stack([x_init, y_init])


fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
speed = 5
def update_springs(canvas):
    global frame_tracker

    frame_tracker = ((frame_tracker + speed) % steps)
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    # Since x is actually the displacement
    dots_graphic.data[:, 0] = (x[frame_tracker] + x_init).astype(np.float32)

fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

## Train

Normal method

In [136]:
X = np.column_stack((x, v))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [137]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [138]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, u_val, -u_val)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9134 99.48%


In [ ]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)

print("\nLearned Bias (Intercept):")
print(ridge_model.intercept_)

print("\nBest Alpha:")
print(ridge_model.alpha_)

Learned Weights (Coefficients):
[ 19.38097394 -54.68918231  39.9599643   -4.87314788   0.7139385
  -0.89417127  -2.7300877    1.00518301   3.25966376  -1.0413492
  -4.19630525  -3.22278617  13.20329226  -9.72115939   5.69955348
   0.5746495   -2.21195174  -0.33574023   1.20490054   0.60664784]

Learned Bias (Intercept):
-0.0006813720486278663

Best Alpha:
0.1


In [140]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()

Scale weights since due to damping the last spring moves less

In [141]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [142]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [143]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, u_val, -u_val)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9156 99.97%


In [144]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)

# 2. Print the bias (intercept) term
print("\nLearned Bias (Intercept):")
print(ridge_model.intercept_)

# 3. Print the best alpha chosen by RidgeCV
print("\nBest Alpha:")
print(ridge_model.alpha_)

Learned Weights (Coefficients):
[  0.60574569 -10.6939231   10.28391654  -0.41380001  -1.46792205
   1.5278263   -1.17488712  -0.22021255   0.91856189  -0.04995501
  -2.55776321   1.70227778   6.58897205  -2.66972347   6.52848945
  -1.26770081   0.4917859    0.01764035  -0.04776241   0.10648586]

Learned Bias (Intercept):
-0.0002443119270239177

Best Alpha:
0.1


In [145]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()

# Lots of Springs with u -1 to 1

In [ ]:
time = 1000  # Total seconds
dt = 0.005
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

N = 100

m_diag = np.random.uniform(0.1, 1.0, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 1, size=N))

k_vals = np.random.uniform(.5, 20, N - 1)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals
K[idx + 1, idx] = -k_vals
k_padded = np.pad(k_vals, (1, 1), mode="constant")
K[np.arange(N), np.arange(N)] = k_padded[:-1] + k_padded[1:]

t = np.linspace(0, time, steps)

u_val = 10
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [13]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        # We apply the force of u to the first spring
        u_force = np.zeros(N)
        u_force[0] = u[i - 1]

        # Velocity Verlet
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        # Now using what values we just found we calc v
        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        # Do v[i - 1] + acc * dt to get v[i]
        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [14]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [15]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [16]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [17]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, u_val, -u_val)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9270 99.96%


# Spring Loop with u -1 to 1

In [187]:
time = 1000  # Total seconds
dt = 0.005
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

N = 10

m_diag = np.random.uniform(0.1, 1.0, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 0.5, size=N))

k_vals = np.random.uniform(1, 8, N)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals[:-1]
K[idx + 1, idx] = -k_vals[:-1]
K[np.arange(N), np.arange(N)] = k_vals + np.roll(k_vals, shift=1)
K[0, N - 1] = -k_vals[-1]
K[N - 1, 0] = -k_vals[-1]

t = np.linspace(0, time, steps)

u_val = 3
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [188]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        # We apply the force of u to the first spring
        u_force = np.zeros(N)
        u_force[0] = u[i - 1]

        # Velocity Verlet
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        # Now using what values we just found we calc v
        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        # Do v[i - 1] + acc * dt to get v[i]
        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [189]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [209]:
radius = 5.0
thetas = np.linspace(0, 2 * np.pi, N, endpoint=False)
x_init = (radius * np.cos(thetas)).astype(np.float32)
y_init = (radius * np.sin(thetas)).astype(np.float32)

tang_vectors = np.column_stack([-y_init, x_init])
tang_norm = np.linalg.norm(tang_vectors, axis=1, keepdims=True)
tang_unit = tang_vectors / tang_norm

dots_data = np.column_stack([x_init, y_init])

fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
speed = 5
def update_springs(canvas):
    global frame_tracker

    frame_tracker = ((frame_tracker + 1) % steps) + speed
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    dots_graphic.data[:, 0] = (x_init + x[frame_tracker] * tang_unit[:, 0]).astype(np.float32)
    dots_graphic.data[:, 1] = (y_init + x[frame_tracker] * tang_unit[:, 1]).astype(np.float32)

fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

In [198]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [199]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [200]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, u_val, -u_val)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9129 99.51%


In [201]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)

# 2. Print the bias (intercept) term
print("\nLearned Bias (Intercept):")
print(ridge_model.intercept_)

# 3. Print the best alpha chosen by RidgeCV
print("\nBest Alpha:")
print(ridge_model.alpha_)

Learned Weights (Coefficients):
[ -0.83463312  19.12056699 -20.3323602   26.27546946  -1.82104512
  -1.75410288   4.63950024  -9.64181052  32.13535004 -20.56129464
  -5.18905764 -25.16316787  16.93460307 -24.3254457   -2.6356349
   5.5112016   -3.06209344   3.86745758 -20.45503692  11.96513795]

Learned Bias (Intercept):
-0.002741087993177283

Best Alpha:
0.1


In [202]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()

# Spring Loop with u from -1 to 1 with force applied to all

In [203]:
time = 1000  # Total seconds
dt = 0.005
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

N = 10

m_diag = np.random.uniform(0.1, 1.0, size=N)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(np.random.uniform(0.1, 0.5, size=N))

k_vals = np.random.uniform(1, 8, N)
K = np.zeros((N, N))
idx = np.arange(N - 1)
K[idx, idx + 1] = -k_vals[:-1]
K[idx + 1, idx] = -k_vals[:-1]
K[np.arange(N), np.arange(N)] = k_vals + np.roll(k_vals, shift=1)
K[0, N - 1] = -k_vals[-1]
K[N - 1, 0] = -k_vals[-1]

t = np.linspace(0, time, steps)

u_val = 3
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [204]:
@njit
def run_simulation(steps, dt, u, M_INV, K, C, N):
    x = np.zeros((steps, N))
    v = np.zeros((steps, N))

    acc = np.zeros(N)

    for i in range(1, steps):
        # We apply the force of u to the first spring
        u_force = np.ones(N) * u[i - 1]

        # Velocity Verlet
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + u_force)

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        # Now using what values we just found we calc v
        u_force_next = np.zeros(N)
        u_force_next[0] = u[i]

        # Do v[i - 1] + acc * dt to get v[i]
        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + u_force_next)

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [205]:
x, v = run_simulation(steps, dt, u, M_INV, K, DAMP, N)

In [210]:
radius = 5.0
thetas = np.linspace(0, 2 * np.pi, N, endpoint=False)
x_init = (radius * np.cos(thetas)).astype(np.float32)
y_init = (radius * np.sin(thetas)).astype(np.float32)

tang_vectors = np.column_stack([-y_init, x_init])
tang_norm = np.linalg.norm(tang_vectors, axis=1, keepdims=True)
tang_unit = tang_vectors / tang_norm

dots_data = np.column_stack([x_init, y_init])

fig = fpl.Figure()
dots_graphic = fig[0, 0].add_scatter(data=dots_data, sizes=15, colors="magenta")

frame_tracker = 0
speed = 5


def update_springs(canvas):
    global frame_tracker

    frame_tracker = ((frame_tracker + 1) % steps) + speed
    print(f"Frame: {frame_tracker}/{steps}", end="\r")

    dots_graphic.data[:, 0] = (x_init + x[frame_tracker] * tang_unit[:, 0]).astype(
        np.float32
    )
    dots_graphic.data[:, 1] = (y_init + x[frame_tracker] * tang_unit[:, 1]).astype(
        np.float32
    )


fig.add_animations(update_springs)
fig.show()

RFBOutputContext()

In [211]:
X = np.column_stack((x, v))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [212]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [213]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, u_val, -u_val)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9174 99.94%


In [214]:
print("Learned Weights (Coefficients):")
print(ridge_model.coef_)

# 2. Print the bias (intercept) term
print("\nLearned Bias (Intercept):")
print(ridge_model.intercept_)

# 3. Print the best alpha chosen by RidgeCV
print("\nBest Alpha:")
print(ridge_model.alpha_)

Learned Weights (Coefficients):
[ -8.20208967  29.73209086  -8.85751244   6.14659488   9.04826293
 -15.63590401  30.58883509 -12.03910619   8.1822149   -4.4324998
  -5.6185978   33.25679867 -11.04708847  17.79550349  -1.56875016
  -9.67393557  20.62918795 -14.78826439   7.3186072   -6.91658494]

Learned Bias (Intercept):
0.004375655030868415

Best Alpha:
0.1


In [215]:
weights = ridge_model.coef_
labels = [
    f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
]

fig = go.Figure(
    data=[
        go.Bar(
            x=labels,
            y=weights,
            marker_color=np.where(
                weights >= 0, "royalblue", "firebrick"
            ),  # Blue for +, Red for -
        )
    ]
)

fig.update_layout(
    title="Reservoir Node Contribution (Feature Weights)",
    xaxis_title="Spring/Mass Node",
    yaxis_title="Weight Value",
    template="plotly_white",
)

fig.show()